# 03 · Create mixed spots with known cell-class proportions

In this tutorial, you will simulate expression for individual cells, then combine nearby cells into larger spatial spots.
Because each source cell has a known class, you can calculate the exact class proportions in every mixed spot.
These proportions provide ground truth for evaluating a deconvolution method.

Use **Zhuang-ABCA-1.007** with its `cell_class` labels. We use the **0.25** resolution setting from Study 03.
See [setup](README.md#setup) for the input file.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import FEAST

TUTORIAL = Path.cwd() if Path.cwd().name == "tutorial" else Path.cwd() / "tutorial"
sys.path.insert(0, str(TUTORIAL))
from _utils import data_root, load_counts, gene_summary, gene_values, spatial_panel
DATA = data_root()

OUT = TUTORIAL / "outputs" / "03"
OUT.mkdir(parents=True, exist_ok=True)
print("FEAST", FEAST.__version__)

In [ ]:
from FEAST.deconvolution import create_deconvolution_benchmark_data
reference = load_counts(DATA / "merfish/Zhuang-ABCA-1.007.h5ad", "cell_class")

### 1. Simulate expression for individual cells

FEAST first generates a new count matrix at the original cell locations.
We will then sum these counts when combining cells into spots.

Set `clip_overshoot_factor=0` to keep the simulated counts as integers.
This disables a final clipping step that can otherwise produce fractional values.

In [ ]:
%%capture --no-stderr
cells = FEAST.simulate(
    reference, seed=2026, parameter_mode="hungarian", spatial_mode="reference_rank",
    assignment_solver="scipy", assignment_blocks=False,
    clip_overshoot_factor=0.0, verbose=False,
)

### 2. Combine cells into larger spots

FEAST places a coarser grid over the tissue and assigns each cell to its nearest grid center.
It then sums the assigned cells' expression counts at each center.

`downsampling_factor=0.25` requests roughly one center for every four input cells.
The final number depends on the tissue shape, so it will not necessarily be exactly one quarter of the input size.
We retain the Study 03 tissue-boundary setting, `alpha=0.01`, for these coordinates.

In [ ]:
spots = create_deconvolution_benchmark_data(
    cells, downsampling_factor=0.25, grid_type="hexagonal",
    cell_type_key="cell_class", alpha=0.01,
)
truth = pd.DataFrame(spots.obsm["cell_type_proportions"],
                     index=spots.obs_names, columns=spots.uns["cell_type_names"])
n_cells = np.bincount(spots.uns["spot_assignments"], minlength=spots.n_obs)
spots.obs["n_source_cells"] = n_cells
np.testing.assert_allclose(truth.sum(axis=1), (n_cells > 0).astype(float))
np.testing.assert_allclose(np.asarray(spots.X.sum(axis=0)), np.asarray(cells.X.sum(axis=0)))
print(f"{cells.n_obs:,} cells → {spots.n_obs:,} centers; {(n_cells == 0).sum()} empty centers")

### 3. Plot the known cell-class proportions

A proportion of 0.6 means that 60% of the cells assigned to a spot belong to that class.
It describes the number of cells, not their share of RNA: different cells can contribute different transcript counts.

Plot the three most abundant classes to keep the figure readable.
The exported table still contains every class.

In [ ]:
shown = reference.obs["cell_class"].value_counts().head(3).index
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), layout="constrained")
for ax, label in zip(axes, shown):
    spatial_panel(ax, spots.obsm["spatial"], truth[label], str(label), vmax=1)
plt.show()
truth.head().round(3)

In [ ]:
spots.write_h5ad(OUT / "spatial_mixtures.h5ad")
truth.to_csv(OUT / "cell_class_truth.csv")
reference.write_h5ad(OUT / "single_cell_reference.h5ad")

**How to read the plots:** each panel shows one class, from 0 (absent) to 1 (all assigned cells).
A spot with positive values in more than one panel contains a mixture of the displayed classes.
Empty grid centers have zeros across the entire proportion table.

Use `spatial_mixtures.h5ad` and the labeled reference as inputs to a method such as RCTD or Cell2location.
Keep the known proportions for evaluating its predictions. Study 03 contains the full fitting and scoring examples.

### Saved example

![Zhuang-ABCA-1.007: cell-class proportions in the simulated 0.25-resolution mixtures.](assets/03.png)

Zhuang-ABCA-1.007: cell-class proportions in the simulated 0.25-resolution mixtures.

This image comes from an earlier reproduction run. It is not a new result from this notebook. A fresh run may differ with the software version and environment.

<details>
<summary>Image sources</summary>

Rendered on 2026-09-04 from these existing files in `FEAST_reproduce`, using the plotting code above:

- `03_deconvolution/data/local/Zhuang-ABCA-1.007.h5ad`
- `03_deconvolution/outputs/cell_class_rerun_20260810_v1/simulations/007/resolution_0.25.h5ad`

The source files were read without rerunning the simulations. These images do not show a new execution of the notebook.

</details>